# Multimodal Meme Understanding: Image, Text, or Both?
### Deep Learning Final Project
Dataset: Memotion 7k (SemEval-2020 Task 8, Task A: 3-class sentiment).

Pipeline (see `docs/project-roadmap.md`):
1. Environment setup & Google Drive mount
2. Dataset download & verification
3. Data preparation & leakage control (Hold-out split)
4. Text-only Baseline (BERT) - Role B
5. Image-only Baseline (ResNet50) - Role C
6. Multimodal Fusion (Concat & Cross-Attention) - Role D
7. Evaluation, ablation & error analysis - Role E & F
8. Demo inference

## 1. Environment Setup & Google Drive Mount

In [12]:
from google.colab import drive
drive.mount('/content/drive')
import os

BRANCH = os.environ.get('BRANCH', 'main')
REPO_URL = 'https://github.com/arcrek/dl2026.git'
ROOT = '/content/drive/MyDrive/dl2026_memotion'
os.environ['DATA_DIR'] = '/content/data/memotion'
os.environ['FEATURE_DIR'] = f'{ROOT}/features'
os.environ['RESULTS_DIR'] = f'{ROOT}/results'

for d in ('features', 'results'):
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)

if os.path.exists('/content/repo'):
    !git -C /content/repo fetch && git -C /content/repo checkout $BRANCH && git -C /content/repo pull
else:
    !git clone -b $BRANCH $REPO_URL /content/repo
%cd /content/repo

!pip install -q transformers scikit-learn pyarrow huggingface_hub pytest


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Already up to date.
/content/repo


## 2. Dataset Download & Verification
> **Note:** For the Text-only Baseline (Role B), you can pass `--jsonl-only` to `scripts/download_data.sh` to download text metadata in seconds without fetching 1.5GB of images.

In [13]:
TAR = f'{ROOT}/memotion.tar'
if os.path.exists(TAR) and not os.path.exists('/content/data/memotion/train.jsonl'):
    !mkdir -p /content/data && tar -xf "$TAR" -C /content/data

!bash scripts/download_data.sh
!python src/data.py --images

if not os.path.exists(TAR):
    !tar -cf "$TAR" -C /content/data memotion


train.jsonl already exists, skipping download...
validation.jsonl already exists, skipping download...
test.jsonl already exists, skipping download...
Dataset ready in /content/data/memotion
train       n= 5593 labels={0: 518, 1: 1762, 2: 3313} OK
  All 5593 images verified for train.
validation  n=  699 labels={0: 59, 1: 221, 2: 419} OK
  All 699 images verified for validation.
test        n=  700 labels={0: 54, 1: 218, 2: 428} OK
  All 700 images verified for test.
Holdout: fit=5034 holdout=559 dist={0: 54, 1: 165, 2: 340}


## 3. Data Preparation & Leakage Control

In [14]:
import json, os, sys
from collections import Counter
sys.path.insert(0, 'src')
import data

D, F = data.DATA_DIR, data.FEATURE_DIR
splits = {s: data.load_jsonl(s, D) for s in data.SPLITS}
train = splits['train']
by_id = {r['id']: r for r in train}

fit_ids, hold_ids = data.save_holdout(D)
with open(os.path.join(F, 'train_holdout.json')) as f:
    meta = json.load(f)
print(f"grouping={meta['grouping']}  fit={len(fit_ids)}  holdout={len(hold_ids)}")
print('class_weights (fit):', [round(w, 3) for w in meta['class_weights']])
for name, ids in (('fit', fit_ids), ('holdout', hold_ids)):
    c = Counter(by_id[i]['label'] for i in ids)
    print(f"{name:8s}", {data.CLASS_NAMES[k]: round(c[k] / len(ids), 3) for k in range(3)})

gid, _ = data.group_ids(train, D)
overlap = {gid[i] for i in fit_ids} & {gid[i] for i in hold_ids}
assert not overlap, f'{len(overlap)} groups straddle fit/holdout'
assert not set(fit_ids) & set(hold_ids)

tr_txt = {data._norm_text(r['text']) for r in train} - {''}
for s in ('validation', 'test'):
    dup = sum(data._norm_text(r['text']) in tr_txt for r in splits[s])
    print(f"{s}: {dup}/{len(splits[s])} captions also appear in train")
assert all('text_ocr' not in r for r in train), 'text_ocr must not be loaded'
print('Leakage checks passed: 0 groups shared between fit and hold-out.')


grouping=text+image  fit=5034  holdout=559
class_weights (fit): [3.616, 1.051, 0.564]
fit      {'negative': 0.092, 'neutral': 0.317, 'positive': 0.591}
holdout  {'negative': 0.097, 'neutral': 0.295, 'positive': 0.608}
validation: 8/699 captions also appear in train
test: 10/700 captions also appear in train
Leakage checks passed: 0 groups shared between fit and hold-out.


## 4. Text-only Baseline (BERT) - Role B
Fine-tune `bert-base-uncased` with Differential Learning Rate across seeds 0, 1, 2.
The training module is decoupled into 4 distinct functions:
1. Loss & Optimizer configuration (`build_loss_and_optimizer`)
2. Evaluation function (`evaluate`)
3. Single-epoch training step (`train_epoch`)
4. Multi-epoch training loop (`train_loop`)

### 4.1 Loss Function & Optimizer Setup

In [15]:
import sys
sys.path.insert(0, 'src')
import torch
import torch.nn as nn
from text import BertMemeClassifier, build_loss_and_optimizer

print('Part 1: Loss & Optimizer module loaded.')


Part 1: Loss & Optimizer module loaded.


### 4.2 Evaluation Function

In [16]:
from text import evaluate

print('Part 2: Evaluation module loaded.')


Part 2: Evaluation module loaded.


### 4.3 Training Step (Single Epoch)

In [17]:
from text import train_epoch

print('Part 3: Single-epoch training step module loaded.')


Part 3: Single-epoch training step module loaded.


### 4.4 Training Loop & Execution

In [18]:
from text import train_loop

print('Part 4: Multi-epoch training loop module loaded.')


Part 4: Multi-epoch training loop module loaded.


In [19]:
!python src/text.py --seeds 0 1 2 --epochs 5 --batch 32


Starting Text Baseline training on device: CUDA
Config: {'epochs': 5, 'batch': 32, 'lr_backbone': 1.5e-05, 'lr_head': 0.0005, 'max_length': 128, 'workers': 2, 'dropout': 0.2, 'weight_decay': 0.01}
Seeds: [0, 1, 2]

==================== Training Seed 0 ====================
Loading weights: 100% 199/199 [00:00<00:00, 4752.82it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading f

In [20]:
!pytest tests/test_results.py -v


============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/repo
plugins: langsmith-0.14.2, anyio-4.15.1, platformdirs-4.12.2, typeguard-4.6.0
collected 3 items                                                              

tests/test_results.py::test_roundtrip PASSED                             [ 33%]
tests/test_results.py::test_rejects_bad_probs_and_lengths PASSED         [ 66%]
tests/test_results.py::test_config PASSED                                [100%]

============================== 3 passed in 0.13s ===============================


In [21]:
import os, sys
sys.path.insert(0, 'src')
import numpy as np
import pandas as pd
from results import load_results

records = []
for s in [0, 1, 2]:
    try:
        res = load_results('text', s)
    except FileNotFoundError:
        continue
    records.append({
        'Seed': s,
        'Best Epoch (Holdout)': res['extra'].get('best_epoch', 'N/A'),
        'Holdout Macro-F1': round(res['extra'].get('holdout_macro_f1', 0.0), 4),
        'Test Accuracy': round(res['accuracy'], 4),
        'Test Macro-F1': round(res['macro_f1'], 4)
    })

if records:
    df = pd.DataFrame(records)
    if len(records) > 1:
        mean_row = pd.DataFrame([{
            'Seed': 'Mean +/- Std',
            'Best Epoch (Holdout)': '-',
            'Holdout Macro-F1': f"{df['Holdout Macro-F1'].mean():.4f} +/- {df['Holdout Macro-F1'].std():.4f}",
            'Test Accuracy': f"{df['Test Accuracy'].mean():.4f} +/- {df['Test Accuracy'].std():.4f}",
            'Test Macro-F1': f"{df['Test Macro-F1'].mean():.4f} +/- {df['Test Macro-F1'].std():.4f}"
        }])
        summary_df = pd.concat([df, mean_row], ignore_index=True)
    else:
        summary_df = df
    display(summary_df)
else:
    print('No results found. Run training cell above first.')


,Seed,Best Epoch (Holdout),Holdout Macro-F1,Test Accuracy,Test Macro-F1
0,0,2,0.3337,0.3914,0.3283
1,1,5,0.2942,0.3543,0.2988
2,2,1,0.3426,0.55,0.3677
3,Mean ± Std,-,0.3235 ± 0.0258,0.4319 ± 0.1039,0.3316 ± 0.0346



=== DÒNG KẾT QUẢ ĐỂ PASTE VÀO BÁO CÁO MỤC 5 ===
Text-only (BERT) | Accuracy: 0.4319 ± 0.1039 | Macro-F1: 0.3316 ± 0.0346


In [22]:
try:
    from google.colab import files
    for s in [0, 1, 2]:
        f_path = f"results/text_seed{s}.json"
        if os.path.exists(f_path):
            files.download(f_path)
except Exception:
    pass


## 5. Image-only Baseline (ResNet50)


## 6. Multimodal Fusion (Concat & Cross-Attention)


## 7. Evaluation, Ablation & Error Analysis


## 8. Demo Inference
